# 🎬 Supernan Hindi Dubbing Pipeline

**Zero-cost pipeline: English video → Hindi dubbed video with lip sync**

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

Pipeline stages:
1. Install dependencies
2. Clone VideoReTalking & GFPGAN
3. Upload & extract 15-second clip
4. Whisper transcription
5. IndicTrans2 Hindi translation
6. Coqui XTTS v2 voice cloning
7. Audio speed adjustment
8. VideoReTalking lip-sync
9. GFPGAN face enhancement
10. Download output

In [1]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import subprocess, torch
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else 'No NVIDIA GPU.')
except FileNotFoundError:
    print('No nvidia-smi (Mac/CPU mode).')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


No nvidia-smi (Mac/CPU mode).
Device: cpu


In [2]:
# ── Cell 2: Install System Deps ───────────────────────────────────────────────
!apt-get install -y ffmpeg wget unzip -q
print('✓ ffmpeg installed')

zsh(45830) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(45830) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh:1: command not found: apt-get
✓ ffmpeg installed


In [3]:
# ── Cell 3: Install Python packages ──────────────────────────────────────────
# PyTorch with CUDA (Colab default, but ensuring compatibility)
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118

# Core pipeline deps
!pip install -q openai-whisper TTS transformers sentencepiece sacremoses
!pip install -q git+https://github.com/VarunGumma/IndicTransTokenizer.git
!pip install -q pydub librosa soundfile deep-translator
!pip install -q basicsr facexlib gfpgan realesrgan
print("✓ All packages installed")


zsh(45840) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(45840) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
python(45840) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
Python(45840) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
Note: you may need to restart the kernel to use updated packages.
zsh(45841) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(45841) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory 

In [4]:
# ── Cell 4: Clone Repos ───────────────────────────────────────────────────────
import os

# VideoReTalking
if not os.path.exists('VideoReTalking'):
    !git clone -q https://github.com/vinthony/video-retalking VideoReTalking
    %pip install -q -r VideoReTalking/requirements.txt
print('✓ VideoReTalking ready')

# GFPGAN
if not os.path.exists('GFPGAN'):
    !git clone -q https://github.com/TencentARC/GFPGAN GFPGAN
    %pip install -q basicsr facexlib gfpgan
    %cd GFPGAN
    !python setup.py develop -q
    %cd ..
print('✓ GFPGAN ready')

✓ VideoReTalking ready
✓ GFPGAN ready


In [5]:
# ── Cell 5: Download VideoReTalking Weights ──────────────────────────────────
import os
os.makedirs("models/VideoReTalking", exist_ok=True)
!wget -q https://github.com/vinthony/video-retalking/releases/download/v0.0.1/30_net_G.pth -O models/VideoReTalking/30_net_G.pth
!wget -q https://github.com/vinthony/video-retalking/releases/download/v0.0.1/BFM.zip -O models/VideoReTalking/BFM.zip
!unzip -qo models/VideoReTalking/BFM.zip -d models/VideoReTalking/
print("✓ VideoReTalking weights ready")


zsh(45886) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(45886) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
wget(45886) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(45889) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(45889) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
wget(45889) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(47139) MallocStackLogging: process is not in a debuggable envi

In [6]:
# ── Cell 6: Download GFPGAN Weights ──────────────────────────────────────────
os.makedirs('GFPGAN/experiments/pretrained_models', exist_ok=True)
gfpgan_weight = 'GFPGAN/experiments/pretrained_models/GFPGANv1.4.pth'
if not os.path.exists(gfpgan_weight):
    !wget -q --show-progress -O {gfpgan_weight} https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth
print('✓ GFPGAN weights ready')

✓ GFPGAN weights ready


In [7]:
# ── Cell 7: Download Input Video from Google Drive ──────────────────────────
import os, subprocess

# ── Paste your Google Drive share link here ───────────────────────────────────
GDRIVE_URL = 'https://drive.google.com/file/d/1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW/view?usp=sharing'

INPUT_VIDEO = 'supernan_training.mp4'

if not os.path.exists(INPUT_VIDEO):
    print('Downloading video from Google Drive...')
    # Install gdown if needed
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    gdown.download(url=GDRIVE_URL, output=INPUT_VIDEO, fuzzy=True)

if os.path.exists(INPUT_VIDEO):
    size_mb = os.path.getsize(INPUT_VIDEO) / 1e6
    print(f'✓ Video ready: {INPUT_VIDEO} ({size_mb:.1f} MB)')
else:
    print('⚠️  Download failed. If the file is restricted, share it as "Anyone with the link"')
    print('   Or place the video manually in:', os.getcwd())


✓ Video ready: supernan_training.mp4 (621.9 MB)


In [8]:
# ── Cell 8: Configuration ────────────────────────────────────────────────────
# ⬇️ Change these if you want a different segment
START_SEC = 30
END_SEC   = 50

os.makedirs('workspace', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('models', exist_ok=True)

print(f'Processing segment: {START_SEC}s – {END_SEC}s ({END_SEC - START_SEC}s clip)')

Processing segment: 30s – 50s (20s clip)


In [9]:
# ── Cell 9: Stage 1 — Extract Clip ───────────────────────────────────────────
CLIP = 'workspace/clip.mp4'
duration = END_SEC - START_SEC

!ffmpeg -y -ss {START_SEC} -i "{INPUT_VIDEO}" -t {duration} \
    -c:v libx264 -c:a aac -preset fast -crf 18 {CLIP} -loglevel warning

from IPython.display import Video
print('✓ Clip extracted')
Video(CLIP, width=640)

zsh(47178) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(47178) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(47178) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
✓ Clip extracted


In [10]:
# ── Cell 10: Stage 2 — Extract Audio ─────────────────────────────────────────
import os
if "CLIP" not in globals(): CLIP = "workspace/clip.mp4"
AUDIO_RAW = "workspace/clip_audio.wav"
AUDIO_16K = "workspace/clip_audio_16k.wav"
AUDIO_REF = "workspace/clip_audio_ref44k.wav"

print("Extracting audio tracks...")
# 1. Raw audio
get_ipython().system(f"ffmpeg -y -i {CLIP} -vn -acodec pcm_s16le -ar 44100 {AUDIO_RAW} -loglevel warning")
# 2. 16k for Whisper
get_ipython().system(f"ffmpeg -y -i {AUDIO_RAW} -ac 1 -ar 16000 {AUDIO_16K} -loglevel warning")
# 3. 44k Reference for XTTS
get_ipython().system(f"ffmpeg -y -i {AUDIO_RAW} -ac 1 -ar 44100 {AUDIO_REF} -loglevel warning")
print("✓ Audio extracted (Raw, 16k, 44k Ref)")


Extracting audio tracks...
zsh(47329) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(47329) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(47329) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(47330) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(47330) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(47330) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
[aist#0:0/pcm_s16le @ 0xab0c18180] 

In [11]:
# ── Cell 11: Stage 3 — Transcribe with Whisper ───────────────────────────────
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import whisper

print('Loading Whisper medium model (downloading ~1.5 GB on first run)...')
model = whisper.load_model('medium')  # 'small' is faster; 'large' is more accurate

result = model.transcribe(AUDIO_16K, language='en', word_timestamps=True, verbose=False)
ENGLISH_TEXT = result['text'].strip()

with open('workspace/transcript_en.txt', 'w') as f:
    f.write(ENGLISH_TEXT)

print(f'✓ Transcript:\n{ENGLISH_TEXT}')

Loading Whisper medium model (downloading ~1.5 GB on first run)...


/Users/vikashkumar/Desktop/Supernan/venv/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████| 2001/2001 [00:48<00:00, 41.16frames/s]

✓ Transcript:
Clean your body, take bath, put deodorant, put a cloth Why do you do all these? Because you are always around your child Your child is always around you So if you smell your child's body or smell their breath Children will not like it They will not come to you


In [12]:
# ── Cell 12: Stage 4 — Translate to Hindi (IndicTrans2) ──────────────────────
import torch, os
if "DEVICE" not in globals(): DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if "ENGLISH_TEXT" not in globals():
    if os.path.exists("workspace/transcript_en.txt"):
        with open("workspace/transcript_en.txt", "r") as f: ENGLISH_TEXT = f.read().strip()
        print("✓ ENGLISH_TEXT recovered from disk")
    else: raise NameError("ENGLISH_TEXT missing. Run Cell 11.")

def translate_indictrans2(text):
    try:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        from IndicTransTokenizer import IndicProcessor
        MODEL = "ai4bharat/indictrans2-en-indic-1B"
        print("Loading IndicTrans2...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, trust_remote_code=True).to(DEVICE)
        ip = IndicProcessor(inference=True)
        sentences = [s.strip() for s in text.split(".") if s.strip()]
        batch = ip.preprocess_batch(sentences, src_lang="eng_Latn", tgt_lang="hin_Deva")
        inputs = tokenizer(batch, truncation=True, padding="longest", return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, num_beams=5, max_length=256)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        postprocessed = ip.postprocess_batch(decoded, lang="hin_Deva")
        return " ".join(postprocessed)
    except Exception as e:
        print(f"IndicTrans2 failed: {e}")
        return None

print("Attempting Hindi translation...")
HINDI_TEXT = translate_indictrans2(ENGLISH_TEXT)

if not HINDI_TEXT:
    print("⚠️ IndicTrans2 failed. Attempting Google Translate fallback...")
    try:
        from deep_translator import GoogleTranslator
        HINDI_TEXT = GoogleTranslator(source="en", target="hi").translate(ENGLISH_TEXT)
        print("✓ Google Translate success!")
    except Exception as ge:
        print(f"❌ Google Translate failed: {ge}. Using English as last resort.")
        HINDI_TEXT = ENGLISH_TEXT

with open("workspace/translation_hi.txt", "w", encoding="utf-8") as f: f.write(HINDI_TEXT)
print(f"✓ Hindi translation completed.")
print(f"Text: {HINDI_TEXT[:100]}...")


Attempting Hindi translation...
IndicTrans2 failed: No module named 'IndicTransTokenizer'
⚠️ IndicTrans2 failed. Attempting Google Translate fallback...
✓ Google Translate success!
✓ Hindi translation completed.
Text: अपना शरीर साफ़ करो, नहाओ, डियोडरेंट लगाओ, कपड़ा डालो तुम ये सब क्यों करते हो? क्योंकि आप हमेशा अपने ...


In [13]:
# ── Cell 13: Stage 5 — Coqui XTTS v2 Voice Cloning ──────────────────────────
import os, re, torch
from TTS.api import TTS

# Recovery & Guards
if "DEVICE" not in globals(): DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if "AUDIO_REF" not in globals(): AUDIO_REF = "workspace/clip_audio_ref44k.wav"
if "HINDI_TEXT" not in globals() or not HINDI_TEXT.strip():
    if os.path.exists("workspace/translation_hi.txt"):
        with open("workspace/translation_hi.txt", "r", encoding="utf-8") as f: HINDI_TEXT = f.read().strip()
    if not globals().get("HINDI_TEXT"):
        if "ENGLISH_TEXT" in globals(): HINDI_TEXT = ENGLISH_TEXT
        else: raise ValueError("HINDI_TEXT empty and no fallback found.")

# 🛡️ PyTorch 2.6+ Unpickling Fix (Comprehensive)
try:
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts.xtts_audio_config import XttsAudioConfig
    import torch.serialization
    torch.serialization.add_safe_globals([XttsConfig, XttsAudioConfig])
    print("✓ Added XTTS configs to PyTorch safe globals")
except Exception as e:
    print(f"⚠️ Could not add safe globals: {e}")

# Final fallback: Monkeypatch torch.load to allow non-weights_only loading
if not hasattr(torch.load, "__supernan_patch__"):
    orig_load = torch.load
    def patched_load(*args, **kwargs):
        if "weights_only" not in kwargs: kwargs["weights_only"] = False
        return orig_load(*args, **kwargs)
    patched_load.__supernan_patch__ = True
    torch.load = patched_load
    print("✓ Applied torch.load security bypass monkeypatch")

os.environ["COQUI_TOS_AGREED"] = "1"
TTS_RAW = "workspace/tts_raw.wav"

print("Loading Coqui XTTS v2...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEVICE)

print(f"Synthesizing Hindi audio...")
tts.tts_to_file(text=HINDI_TEXT, speaker_wav=AUDIO_REF, language="hi", file_path=TTS_RAW)
print(f"✓ Synthesized: {TTS_RAW}")


/Users/vikashkumar/Desktop/Supernan/venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Loading Coqui XTTS v2...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL TTS.tts.models.xtts.XttsAudioConfig was not an allowed global by default. Please use `torch.serialization.add_safe_globals([TTS.tts.models.xtts.XttsAudioConfig])` or the `torch.serialization.safe_globals([TTS.tts.models.xtts.XttsAudioConfig])` context manager to allowlist this global if you trust this class/function.

Check the documentation of to

In [ ]:
# ── Cell 14: Stage 6 — Adjust Audio Speed ────────────────────────────────────
import subprocess, os
def get_duration(path):
    if not os.path.exists(path): return 0.0
    r = subprocess.run(["ffprobe","-v","quiet","-show_entries","format=duration","-of","csv=p=0",path],
                       capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return 0.0

if "CLIP" not in globals(): CLIP = "workspace/clip.mp4"
if "TTS_RAW" not in globals(): TTS_RAW = "workspace/tts_raw.wav"
TTS_ADJ = "workspace/tts_adjusted.wav"

def build_atempo(ratio):
    filters = []
    r = ratio
    while r < 0.5: filters.append("atempo=0.5"); r /= 0.5
    while r > 2.0: filters.append("atempo=2.0"); r /= 2.0
    filters.append(f"atempo={r:.6f}")
    return ",".join(filters)

tts_dur = get_duration(TTS_RAW)
clip_dur = get_duration(CLIP)

if tts_dur > 0 and clip_dur > 0:
    ratio = tts_dur / clip_dur
    atempo = build_atempo(ratio)
    print(f"Adjusting speed (ratio: {ratio:.3f})...")
    get_ipython().system(f"ffmpeg -y -i {TTS_RAW} -filter:a '{atempo},apad' -t {clip_dur} -ar 44100 {TTS_ADJ} -loglevel warning")
    print(f"✓ Audio adjusted to {clip_dur:.2f}s")
else: print("⚠️ Missing files, skipping speed adjustment")


In [ ]:
# ── Cell 15: Stage 7 — VideoReTalking Lip Sync ───────────────────────────────
import sys, os, subprocess
if "CLIP" not in globals(): CLIP = "workspace/clip.mp4"
if "TTS_ADJ" not in globals(): TTS_ADJ = "workspace/tts_adjusted.wav"
LIPSYNC = "workspace/lipsync.mp4"

# Path fixes for VRT
for p in ["VideoReTalking", "VideoReTalking/third_part", "VideoReTalking/third_part/face3d"]:
    path = os.path.abspath(p)
    if path not in sys.path: sys.path.append(path)

print("Running VideoReTalking (Stage 7)...")
res = subprocess.run([sys.executable, "VideoReTalking/inference.py", "--face", CLIP, "--audio", TTS_ADJ, "--outfile", LIPSYNC, "--checkpoint_dir", "models/VideoReTalking"])

if res.returncode != 0:
    print("⚠️ VRT failed, falling back to simple mux.")
    get_ipython().system(f"ffmpeg -y -i {CLIP} -i {TTS_ADJ} -map 0:v -map 1:a -c:v copy -shortest {LIPSYNC} -loglevel warning")

from IPython.display import Video
Video(LIPSYNC, width=640)


In [ ]:
# ── Cell 16: Stage 8 — GFPGAN Face Enhancement ───────────────────────────────
SKIP_ENHANCEMENT = False  # Set to True to skip this slow stage and get output immediately
FINAL = "output/final_dubbed.mp4"
FRAMES_DIR = "workspace/frames_raw"
ENHANCED_DIR = "workspace/gfpgan_out"

import os, subprocess, torch

if SKIP_ENHANCEMENT:
    print("⏩ SKIP_ENHANCEMENT is True. Copying lip-sync video to final output...")
    get_ipython().system(f"cp {LIPSYNC} {FINAL}")
    print(f"✓ Final output (no enhancement): {FINAL}")
else:
    os.makedirs(FRAMES_DIR, exist_ok=True)
    print("Extracting frames...")
    get_ipython().system(f"ffmpeg -y -i {LIPSYNC} -qscale:v 1 -qmin 1 {FRAMES_DIR}/frame_%06d.png -loglevel warning")

    print("Running GFPGAN face restoration...")
    if torch.cuda.is_available():
        PROCESSOR = "cuda"
    elif torch.backends.mps.is_available():
        PROCESSOR = "mps"
    else:
        PROCESSOR = "cpu"
    print(f"Device detected: {PROCESSOR}")
    get_ipython().system(f"python GFPGAN/inference_gfpgan.py --ext png --device {PROCESSOR} -i {FRAMES_DIR} -o {ENHANCED_DIR} -v 1.4 -s 1 --bg_upsampler None")

    fps_r = subprocess.run(["ffprobe","-v","quiet","-select_streams","v:0",
                            "-show_entries","stream=r_frame_rate","-of","csv=p=0", LIPSYNC],
                           capture_output=True, text=True)
    FPS = fps_r.stdout.strip()

    RESTORED = f"{ENHANCED_DIR}/restored_imgs"
    print("Re-encoding final video...")
    get_ipython().system(f"ffmpeg -y -framerate {FPS} -pattern_type glob -i '{RESTORED}/*.png' -i {LIPSYNC} -map 0:v -map 1:a -c:v libx264 -crf 17 -preset slow -c:a aac -shortest {FINAL} -loglevel warning")

    final_dur = get_duration(FINAL)
    print(f"✓ Final output: {FINAL} ({final_dur:.2f}s)")

from IPython.display import Video
Video(FINAL, width=640)


In [ ]:
# ── Cell 17: Download Output ──────────────────────────────────────────────────
import os
try:
    from google.colab import files
    files.download(FINAL)
    print(f"✓ Download started for {FINAL}")
except ImportError:
    print(f"✓ Output saved to: {os.path.abspath(FINAL)}")
    print("   (Local machine detected, skipping browser download)")


---
## 📊 Pipeline Summary

| Stage | Tool | Duration |
|---|---|---|
| Clip Extract | ffmpeg | ~5s |
| Audio Extract | ffmpeg | ~2s |
| Transcription | Whisper medium | ~30s |
| Translation | IndicTrans2 | ~60s |
| Voice Clone | Coqui XTTS v2 | ~3-5 min |
| Speed Adjust | ffmpeg atempo | ~5s |
| Lip Sync | VideoReTalking | ~10-20 min |
| Face Enhance | GFPGAN v1.4 | ~2-3 min |

**Total on free Colab T4: ~15-30 minutes for a 15-second clip**

**Cost: ₹0** ✅